# Queenie — merge correction starter

Hi Queenie! This is your starting point, not a finished model. We load the handover tables, check labels, and measure what happens when we change nothing. Your model choice and training are still yours to do.

**Labels are PROVISIONAL pending group manual review.** Please review a few examples with the group before treating the targets as ground truth. This notebook does not fit a model and never opens test data.

### Keep the boundary clear
- `X` contains only prediction-derived allowlisted features. Reference matches, labels, IDs and split metadata are not model inputs.
- Note features: `start, end, pitch, amplitude, duration, prev_gap, next_gap`; pair features: `gap, pitch_delta, duration_sum`.
- Missing/ambiguous labels are excluded only from this task's training-label diagnostics, never from full end-to-end evaluation.
- Timing targets are absolute seconds; pitch targets are absolute MIDI. Positive classifier labels mean merge the pair / remove the false note.


In [29]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
import sklearn

pd.set_option('display.max_colwidth', None)

# Open from this notebook folder, or run from the project root.
HERE = next(p / 'notebooks' / 'correction_handover' for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'GTSinger_100_unique_sample'] if (p / 'selected_100.csv').exists())
sys.path.insert(0, str(HERE))
import correction_utils as cu
DATA = HERE / '..' / '..' / 'basic_pitch_runs' / 'correction_handover'
notes = pd.read_csv(DATA / 'train_features.csv', dtype={'note_id':str,'recording_id':str,'sample_id':str})
note_targets = pd.read_csv(DATA / 'train_targets.csv', dtype={'note_id':str}).rename(columns={'false_note':'false_label','reference_start':'target_start','reference_end':'target_end','reference_pitch':'target_pitch'})
assert notes['split'].eq('train').all(), 'Only train data belongs in this starter.'
manifest = pd.read_csv(DATA / 'training_recording_manifest.csv')
cu.validate_notes(notes, manifest)
features = pd.read_csv(DATA / 'merge_candidates.csv', dtype={'pair_id':str,'recording_id':str,'note_id_a':str,'note_id_b':str}).rename(columns={'note_id_a':'left_note_id','note_id_b':'right_note_id'})
targets = pd.read_csv(DATA / 'merge_targets.csv', dtype={'pair_id':str}).rename(columns={'merge':'merge_label'})
# Pairs must refer to two real notes from the same train recording.
lookup = notes.set_index('note_id')['recording_id']
for side in ['left_note_id','right_note_id']:
    assert features[side].isin(lookup.index).all(), 'Unknown candidate note'
    assert features[side].map(lookup).eq(features['recording_id']).all(), 'Cross-recording candidate'
assert features.left_note_id.ne(features.right_note_id).all()
TASK = 'merge'
X, joined = cu.training_data(features, targets, TASK)
print('Feature allowlist:', X.columns.tolist())
print('Rows:', len(X), '| target-row coverage: 100%')
display(X.head())


Feature allowlist: ['gap', 'pitch_delta', 'duration_sum']
Rows: 0 | target-row coverage: 100%


,gap,pitch_delta,duration_sum


In [30]:
# === PROPOSED merge v2 (pending group review) ===
# Replaces the empty v1 candidates for THIS notebook only. No shared files are changed.

# --- Candidate rule: uses Basic Pitch's notes only (what the model will see) ---
GAP_MIN = -0.02          # pairs may overlap by up to 0.02 s
GAP_MAX = 0.30           # v1 used 0.12 s, but Basic Pitch never leaves gaps under ~0.128 s
PITCH_DELTA_MAX = 1      # 0 = same pitch only; 1 = up to one semitone apart

# --- Labelling rule: uses training reference notes (answer key only, never model input) ---
MIN_FRACTION_INSIDE = 0.5    # each fragment mostly inside the same real note
MIN_UNION_COVERAGE = 0.6     # together they cover >= 60% of that real note
MAX_PITCH_FROM_REF = 1       # each fragment within 1 semitone of the real note's pitch

refs = pd.read_csv(DATA / 'references_train.csv', dtype={'note_id':str,'recording_id':str})
ref_by_rec = {rid: g[['start','end','pitch']].to_numpy(float) for rid, g in refs.groupby('recording_id')}

def best_ref(n):
    r = ref_by_rec.get(n.recording_id)
    if r is None or len(r) == 0:
        return None, 0.0
    ov = np.clip(np.minimum(r[:,1], n.end) - np.maximum(r[:,0], n.start), 0, None)
    j = int(np.argmax(ov))
    return (j if ov[j] > 0 else None), ov[j] / (n.end - n.start)

def merge_label(a, b):
    ja, fa = best_ref(a); jb, fb = best_ref(b)
    if ja is None or jb is None or min(fa, fb) < MIN_FRACTION_INSIDE:
        return np.nan                                   # unclear -> unresolved
    if ja != jb:
        return 0                                        # two different real notes -> keep separate
    r = ref_by_rec[a.recording_id][ja]
    if max(abs(a.pitch - r[2]), abs(b.pitch - r[2])) > MAX_PITCH_FROM_REF:
        return np.nan
    covered = sum(max(0, min(n.end, r[1]) - max(n.start, r[0])) for n in (a, b))
    return 1 if covered / (r[1] - r[0]) >= MIN_UNION_COVERAGE else np.nan   # same real note -> merge

feature_rows, target_rows = [], []
sorted_notes = notes.sort_values(['recording_id','start','end','pitch'], kind='mergesort')
for rid, g in sorted_notes.groupby('recording_id', sort=False):
    g = g.reset_index(drop=True)
    for i in range(len(g) - 1):
        a, b = g.iloc[i], g.iloc[i + 1]
        gap, pitch_delta = b.start - a.end, b.pitch - a.pitch
        if abs(pitch_delta) > PITCH_DELTA_MAX or not GAP_MIN <= gap <= GAP_MAX:
            continue
        pair_id = f'{rid}:v2p{i:05d}'
        feature_rows.append(dict(pair_id=pair_id, left_note_id=a.note_id, right_note_id=b.note_id,
                                 recording_id=rid, sample_id=a.sample_id, split='train',
                                 gap=gap, pitch_delta=pitch_delta, duration_sum=a.duration + b.duration))
        target_rows.append(dict(pair_id=pair_id, merge_label=merge_label(a, b)))

features = pd.DataFrame(feature_rows)
targets = pd.DataFrame(target_rows)
X, joined = cu.training_data(features, targets, TASK)
print('v1 candidates: 0  ->  v2 candidates:', len(features))
print('v2 labels:', targets.merge_label.map({1:'merge', 0:'keep separate'}).fillna('unresolved').value_counts().to_dict())

v1 candidates: 0  ->  v2 candidates: 1249
v2 labels: {'keep separate': 779, 'unresolved': 305, 'merge': 165}


## Check what we can actually learn from

Target rows must cover every feature ID, even when their task labels are missing. For classification, inspect both class counts before choosing a method. For timing/pitch, the value counts below describe target coverage rather than classifier classes.

In [31]:
label_columns = cu.TARGETS[TASK]
usable = joined[label_columns].notna().all(axis=1)
if 'ambiguous' in joined:
    # CSV accepts True/False or 1/0. Reject unknown encodings rather than guessing.
    ambiguity = joined['ambiguous'].astype(str).str.lower().map({'true':True,'false':False,'1':True,'0':False})
    assert ambiguity.notna().all(), 'ambiguous must be explicit True/False or 1/0'
    usable &= ~ambiguity
print({'total_rows':len(joined), 'usable_task_labels':int(usable.sum()),
       'excluded_task_labels':int((~usable).sum())})
for column in label_columns:
    print(column, 'non-null coverage:', float(joined[column].notna().mean()))
    display(joined.loc[usable,column].value_counts(dropna=False).rename('count'))
labelled = joined.loc[usable].copy()
X_task = X.loc[usable].copy()  # Only these allowlisted columns may reach your future model.
folds = pd.read_csv(DATA / 'folds.csv').set_index('sample_id')['fold']
groups = labelled['sample_id'].copy()
fold_ids = groups.map(folds)
assert fold_ids.notna().all(), 'Missing training-song fold'
print('Labelled examples per song-grouped fold:', fold_ids.value_counts().sort_index().to_dict())
print('Do not fit until the group approves the provisional label version.')


{'total_rows': 1249, 'usable_task_labels': 944, 'excluded_task_labels': 305}
merge_label non-null coverage: 0.755804643714972


merge_label
0.0    779
1.0    165
Name: count, dtype: int64

Labelled examples per song-grouped fold: {0: 175, 1: 199, 2: 213, 3: 186, 4: 171}
Do not fit until the group approves the provisional label version.


## Our first comparison: do nothing

This is a real no-change baseline on usable training task labels, not evidence of validation performance. Empty-label metrics have no useful interpretation. Nothing is fitted here.

In [32]:
if len(labelled) == 0:
    baseline = {'status':'BLOCKED_NO_REVIEWED_OR_PROVISIONAL_TARGETS', 'n':0, 'metrics':None}
else:
    baseline = cu.classification_metrics(labelled[label_columns[0]], np.zeros(len(labelled), dtype=int))
print('No-change task baseline (not a trained model):', baseline)


No-change task baseline (not a trained model): {'n': 944, 'tp': 0, 'fp': 0, 'fn': 165, 'tn': 779, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.8252118644067796}


## Your model goes here — later

Use `X_task` and the separate labels when you are ready. Keep validation separate by song (`sample_id`), choose settings on train/val only, and leave test untouched. Do not infer features from reference alignment. For merge predictions, resolve overlapping pair decisions before producing a complete note table.

In [33]:
# === train and test the merge models fold by fold, with NESTED threshold choice ===
# README rule: "use inner training folds for model selection when reporting outer-fold estimates".
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

fold_all = joined.sample_id.map(folds).to_numpy()          # song-grouped fold of every candidate pair
is_labelled = joined.merge_label.notna().to_numpy()
y_all = joined.merge_label.fillna(-1).astype(int).to_numpy()
all_folds = np.unique(fold_all)

def f05(truth, pred):   # F0.5 = F-score that weights precision more than recall
    m = cu.classification_metrics(truth, pred); p, r = m['precision'], m['recall']
    return 1.25 * p * r / (0.25 * p + r) if p + r else 0.0

# 1) Rule baseline: merge if |pitch difference| <= d and gap <= g. (d, g) chosen on the OTHER folds only.
grid = [(d, g) for d in [0, 1] for g in [0.15, 0.20, 0.25, 0.30]]
def rule(d, g): return ((joined.pitch_delta.abs() <= d) & (joined.gap <= g)).astype(int).to_numpy()
oof = {'Rule baseline': np.zeros(len(joined), dtype=int)}
for f in all_folds:
    train = is_labelled & (fold_all != f)
    best = max(grid, key=lambda dg: f05(y_all[train], rule(*dg)[train]))
    oof['Rule baseline'][fold_all == f] = rule(*best)[fold_all == f]

# 2) Classifiers. For each held-out fold: choose the threshold with INNER folds (training folds only),
#    then train on all training folds and predict EVERY pair in the held-out fold.
models = {
    'Logistic regression': make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000)),
    'Random forest': RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=5,
                                            class_weight='balanced', random_state=0, n_jobs=1),
}
thresholds = np.round(np.arange(0.30, 0.96, 0.05), 2)
def best_threshold(prob, truth):
    return max(thresholds, key=lambda t: f05(truth, (prob >= t).astype(int)))

probability = {name: np.zeros(len(joined)) for name in models}
fold_thresholds = {name: {} for name in models}
for name, base in models.items():
    pred = np.zeros(len(joined), dtype=int)
    for f in all_folds:
        train = is_labelled & (fold_all != f)
        inner_prob = np.zeros(len(joined))
        for k in all_folds[all_folds != f]:                       # inner loop: never sees fold f
            inner_train = train & (fold_all != k)
            inner_prob[fold_all == k] = clone(base).fit(X[inner_train], y_all[inner_train]).predict_proba(X[fold_all == k])[:, 1]
        t = best_threshold(inner_prob[train], y_all[train])
        fold_thresholds[name][int(f)] = float(t)
        test = fold_all == f
        probability[name][test] = clone(base).fit(X[train], y_all[train]).predict_proba(X[test])[:, 1]
        pred[test] = (probability[name][test] >= t).astype(int)
    oof[f'{name} (nested threshold)'] = pred

# Threshold for the FINAL model (used on validation later): chosen on all training out-of-fold probabilities.
chosen_threshold = {name: float(best_threshold(probability[name][is_labelled], y_all[is_labelled])) for name in models}
print('Thresholds chosen inside each outer fold:', fold_thresholds)
print('Threshold frozen for the final model:', chosen_threshold)

# 3) Task metrics on labelled pairs, plus "help vs damage" counts
task_rows = []
for name, pred in oof.items():
    m = cu.classification_metrics(y_all[is_labelled], pred[is_labelled])
    task_rows.append(dict(model=name, merges_proposed=int(pred.sum()),
                          correct_merges=m['tp'], damaging_merges=m['fp'], merges_on_unlabelled=int(pred[~is_labelled].sum()),
                          precision=m['precision'], recall=m['recall'], F1=m['f1'], F05=f05(y_all[is_labelled], pred[is_labelled])))
task_results = pd.DataFrame(task_rows).round(3)
display(task_results)

Thresholds chosen inside each outer fold: {'Logistic regression': {0: 0.55, 1: 0.5, 2: 0.5, 3: 0.5, 4: 0.6}, 'Random forest': {0: 0.45, 1: 0.55, 2: 0.45, 3: 0.5, 4: 0.6}}
Threshold frozen for the final model: {'Logistic regression': 0.6, 'Random forest': 0.5}


,model,merges_proposed,correct_merges,damaging_merges,merges_on_unlabelled,precision,recall,F1,F05
0,Rule baseline,767,125,460,182,0.214,0.758,0.333,0.250
1,Logistic regression (nested threshold),415,70,257,88,0.214,0.424,0.285,0.238
2,Random forest (nested threshold),606,101,386,119,0.207,0.612,0.310,0.239


## Try the shared output interface

The CSV below contains the real original notes unchanged. A future correction CSV is the **complete replacement note set**, not a patch: omitted notes are deletions; for a merge retain one ID and omit absorbed IDs. Never overwrite the source tables.

In [34]:
# Real, unchanged notes: do not filter this with the task-label mask.
unchanged = notes[cu.NOTE_COLUMNS].copy(deep=True)
output = HERE / 'exports' / 'queenie_merge_unchanged.csv'
if output.exists():
    # Safe notebook rerun: verify the existing export instead of overwriting it.
    existing = pd.read_csv(output, dtype={'note_id':str,'recording_id':str})
    pd.testing.assert_frame_equal(cu.validate_notes(existing, manifest), cu.validate_notes(unchanged, manifest), check_dtype=False)
else:
    cu.export_corrections(unchanged, manifest, output)
print('Unchanged-note interface example:', output)


Unchanged-note interface example: c:\Users\queen\Downloads\GTSinger_Correction_Group_Handover (Queenie)\GTSinger_Correction_Group_Handover\GTSinger_100_unique_sample\notebooks\correction_handover\exports\queenie_merge_unchanged.csv


## Full end-to-end scoring (when the evaluator tables are ready)

Use `cu.evaluate_end_to_end(corrected, full_references, evaluation_manifest)` for both the unchanged baseline and future corrected notes. All three tables must describe the same declared train/val scope. `evaluation_manifest` has `recording_id, sample_id, split` and must include recordings with zero predictions. `full_references` has `note_id, recording_id, start, end, pitch`; reference IDs are independent of prediction IDs.

Do **not** build this manifest from a filtered prediction table, or filter references with `usable`, task matches, or ambiguity flags. The manifest loaded above is the complete training recording manifest, including zero-prediction recordings. Reference tables are evaluator-only, never feature inputs.

The shared evaluator reuses `bp_tuning_core.score_notes` and `aggregate`: 50 ms onset, 50-cent pitch, optional offsets (20% or 50 ms), song-pooled macro scores and pooled micro F1. The next cell runs it on every training recording using evaluator-only references. It never filters on task-label availability. No validation or test labels are opened.


In [35]:
full_manifest = pd.read_csv(DATA / 'training_recording_manifest.csv')
full_references = pd.read_csv(DATA / 'references_train.csv')
end_to_end = cu.evaluate_end_to_end(unchanged, full_references, full_manifest)
print('All-training no-correction baseline:', end_to_end['summary'])
assert end_to_end['summary']['recordings'] == len(full_manifest)


All-training no-correction baseline: {'song_macro_f1': 0.26275846642410683, 'song_macro_offset_f1': 0.15386648702425776, 'song_macro_precision': 0.27330092213557544, 'song_macro_recall': 0.2582440522033118, 'micro_f1': 0.26521100683138604, 'songs': 60, 'recordings': 534}


In [36]:
# === Diagnostic: how much could merging help the full transcription? (not a model) ===
def apply_merges(note_table, pairs):
    '''COMPLETE replacement note table. Chains (A~B, B~C) become one note; keeps the first note_id
    and the pitch of the longest fragment.'''
    parent = {nid: nid for nid in note_table.note_id}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for a, b in zip(pairs.left_note_id, pairs.right_note_id):
        parent[find(b)] = find(a)
    work = note_table.assign(group=note_table.note_id.map(find),
                             duration=note_table.end - note_table.start).sort_values('start')
    longest_pitch = work.loc[work.groupby('group').duration.idxmax()].set_index('group').pitch
    merged = work.groupby('group', sort=False).agg(note_id=('note_id', 'first'), recording_id=('recording_id', 'first'),
                                                    start=('start', 'min'), end=('end', 'max'))
    merged['pitch'] = longest_pitch
    return merged.reset_index(drop=True)[cu.NOTE_COLUMNS]

labelled_pairs = features.merge(targets, on='pair_id')
versions = {
    'No change': unchanged,
    'Merge every v2 candidate': apply_merges(unchanged, labelled_pairs),
    'Oracle: merge only label=1 (upper bound)': apply_merges(unchanged, labelled_pairs[labelled_pairs.merge_label == 1]),
}
rows = []
for name, table in versions.items():
    s = cu.evaluate_end_to_end(table, full_references, full_manifest)['summary']
    rows.append(dict(version=name, notes=len(table), F1=s['song_macro_f1'], F1_with_offsets=s['song_macro_offset_f1'],
                     precision=s['song_macro_precision'], recall=s['song_macro_recall']))
display(pd.DataFrame(rows).round(4))

,version,notes,F1,F1_with_offsets,precision,recall
0,No change,7623,0.2628,0.1539,0.2733,0.2582
1,Merge every v2 candidate,6374,0.2385,0.1364,0.2726,0.2161
2,Oracle: merge only label=1 (upper bound),7458,0.2669,0.1616,0.2803,0.2597


In [37]:
# === what does each model do to the FULL transcription? ===
end_rows = []
for name, pred in oof.items():
    corrected = apply_merges(unchanged, joined[pred == 1])
    s = cu.evaluate_end_to_end(corrected, full_references, full_manifest)['summary']
    end_rows.append(dict(version=name, notes=len(corrected), F1=s['song_macro_f1'], F1_with_offsets=s['song_macro_offset_f1'],
                         precision=s['song_macro_precision'], recall=s['song_macro_recall']))
    print('scored:', name)
reference_rows = [dict(version='No change', notes=len(unchanged), F1=0.2628, F1_with_offsets=0.1539),
                  dict(version='Oracle (perfect merging, upper bound)', notes=7458, F1=0.2669, F1_with_offsets=0.1616)]
end_results = pd.concat([pd.DataFrame(reference_rows), pd.DataFrame(end_rows)], ignore_index=True).round(4)
display(end_results)

scored: Rule baseline
scored: Logistic regression (nested threshold)
scored: Random forest (nested threshold)


,version,notes,F1,F1_with_offsets,precision,recall
0,No change,7623,0.2628,0.1539,NaN,NaN
1,"Oracle (perfect merging, upper bound)",7458,0.2669,0.1616,NaN,NaN
2,Rule baseline,6856,0.2457,0.1447,0.2704,0.2296
3,Logistic regression (nested threshold),7208,0.2517,0.1493,0.2694,0.2408
4,Random forest (nested threshold),7017,0.2489,0.1463,0.2707,0.2351


In [38]:
# === save corrected notes + action history for the best model (never overwrites) ===
import time

def merge_actions(note_table, pairs):
    '''Action history required by the README: which note was kept and which IDs were absorbed.'''
    parent = {nid: nid for nid in note_table.note_id}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for a, b in zip(pairs.left_note_id, pairs.right_note_id):
        parent[find(b)] = find(a)
    work = note_table.assign(group=note_table.note_id.map(find)).sort_values('start')
    groups = work.groupby('group', sort=False).note_id.apply(list)
    groups = groups[groups.map(len) > 1]
    return pd.DataFrame({'action': 'merge', 'kept_note_id': groups.map(lambda ids: ids[0]),
                         'absorbed_note_ids': groups.map(lambda ids: ';'.join(ids[1:]))}).reset_index(drop=True)

best_name = max(end_rows, key=lambda r: r['F1'])['version']
best_pred = oof[best_name]
stamp = time.strftime('%Y%m%d_%H%M')
export_dir = HERE / 'exports'

corrected = apply_merges(unchanged, joined[best_pred == 1])
cu.export_corrections(corrected, full_manifest, export_dir / f'queenie_merge_v2_TRAIN_corrected_{stamp}.csv')
merge_actions(unchanged, joined[best_pred == 1]).to_csv(export_dir / f'queenie_merge_v2_TRAIN_actions_{stamp}.csv', index=False)
task_results.to_csv(export_dir / f'queenie_merge_v2_TRAIN_task_results_{stamp}.csv', index=False)
end_results.to_csv(export_dir / f'queenie_merge_v2_TRAIN_end_to_end_{stamp}.csv', index=False)

best_f1 = max(r['F1'] for r in end_rows)
print('Best model on training cross-validation:', best_name, '| F1 =', round(best_f1, 4), '(no change = 0.2628)')
if best_f1 <= 0.2628:
    print('NOTE: no model beat "no change" on training cross-validation.')
print('Saved corrected notes, action history and both results tables in', export_dir)

Best model on training cross-validation: Logistic regression (nested threshold) | F1 = 0.2517 (no change = 0.2628)
NOTE: no model beat "no change" on training cross-validation.
Saved corrected notes, action history and both results tables in c:\Users\queen\Downloads\GTSinger_Correction_Group_Handover (Queenie)\GTSinger_Correction_Group_Handover\GTSinger_100_unique_sample\notebooks\correction_handover\exports


### Validation

In [39]:
# === VALIDATION STEP 1: freeze every choice, train ONCE on all training data, save the models ===
import joblib
y_train, X_train = y_all[is_labelled], X[is_labelled]

frozen_rule = max(grid, key=lambda dg: f05(y_train, rule(*dg)[is_labelled]))
final_models = {name: clone(model).fit(X_train, y_train) for name, model in models.items()}

frozen = {'Rule baseline': f'merge if |pitch difference| <= {frozen_rule[0]} and gap <= {frozen_rule[1]} s'}
for name in final_models:
    frozen[name] = f'merge if probability >= {chosen_threshold[name]}'
frozen['Candidate rule'] = f'neighbouring notes, |pitch difference| <= {PITCH_DELTA_MAX}, gap {GAP_MIN} to {GAP_MAX} s'
frozen['Features'] = ', '.join(X.columns)
frozen['Labels'] = 'proposed merge v2 (automatic, pending group review)'
display(pd.Series(frozen, name='FROZEN before looking at validation').to_frame())

model_path = HERE / 'exports' / f'queenie_merge_v2_models_{time.strftime("%Y%m%d_%H%M")}.joblib'
joblib.dump(dict(models=final_models, thresholds=chosen_threshold, rule=frozen_rule, settings=frozen), model_path)
print('Saved frozen models to', model_path.name)

,FROZEN before looking at validation
Rule baseline,merge if |pitch difference| <= 1 and gap <= 0.15 s
Logistic regression,merge if probability >= 0.6
Random forest,merge if probability >= 0.5
Candidate rule,"neighbouring notes, |pitch difference| <= 1, gap -0.02 to 0.3 s"
Features,"gap, pitch_delta, duration_sum"
Labels,"proposed merge v2 (automatic, pending group review)"


Saved frozen models to queenie_merge_v2_models_20261020_0107.joblib


In [40]:
# === VALIDATION STEP 2: build validation candidates (Basic Pitch predictions only, no answers) ===
val_notes = pd.read_csv(DATA / 'val_features.csv', dtype={'note_id':str,'recording_id':str,'sample_id':str})
assert val_notes['split'].eq('val').all(), 'Validation features only.'

val_rows = []
for rid, g in val_notes.sort_values(['recording_id','start','end','pitch'], kind='mergesort').groupby('recording_id', sort=False):
    g = g.reset_index(drop=True)
    for i in range(len(g) - 1):
        a, b = g.iloc[i], g.iloc[i + 1]
        gap, pitch_delta = b.start - a.end, b.pitch - a.pitch
        if abs(pitch_delta) > PITCH_DELTA_MAX or not GAP_MIN <= gap <= GAP_MAX:
            continue
        val_rows.append(dict(pair_id=f'{rid}:v2p{i:05d}', left_note_id=a.note_id, right_note_id=b.note_id,
                             recording_id=rid, sample_id=a.sample_id, gap=gap, pitch_delta=pitch_delta,
                             duration_sum=a.duration + b.duration))
val_pairs = pd.DataFrame(val_rows)
X_val = val_pairs[X.columns].astype(float)

val_pred = {'Rule baseline': ((val_pairs.pitch_delta.abs() <= frozen_rule[0]) & (val_pairs.gap <= frozen_rule[1])).astype(int).to_numpy()}
val_probability = {}
for name, model in final_models.items():
    val_probability[name] = model.predict_proba(X_val)[:, 1]
    val_pred[f'{name} (threshold {chosen_threshold[name]})'] = (val_probability[name] >= chosen_threshold[name]).astype(int)

print('Validation notes:', len(val_notes), '| recordings:', val_notes.recording_id.nunique(), '| candidate pairs:', len(val_pairs))
display(pd.Series({name: int(p.sum()) for name, p in val_pred.items()}, name='merges proposed on validation').to_frame())

Validation notes: 2329 | recordings: 160 | candidate pairs: 400


,merges proposed on validation
Rule baseline,224
Logistic regression (threshold 0.6),35
Random forest (threshold 0.5),194


In [41]:
# === VALIDATION STEP 3: build the validation answer key from data/val, check it, then score ===
import json, time
ROOT = (HERE / '..' / '..').resolve()
manifest_all = pd.read_csv(DATA / 'recording_manifest.csv', dtype={'recording_id':str,'sample_id':str})
val_manifest = manifest_all[manifest_all.split.eq('val')].reset_index(drop=True)
assert len(val_manifest) and val_manifest.split.eq('val').all()          # test is never read

# Where is each recording's annotation JSON? (recording_manifest json_path, else selected_files.csv export_path)
if 'json_path' in manifest_all:
    json_path = manifest_all.set_index('recording_id').json_path
else:
    files = pd.read_csv(ROOT / 'selected_files.csv', dtype={'recording_id':str})
    json_path = files[files.role.eq('json')].set_index('recording_id').export_path

def parse_reference(rid):
    '''Same policy as the BP tuning notebook: as-annotated note_start/note_end, pitch > 0 only.'''
    duration = float(manifest_all.set_index('recording_id').loc[rid, 'duration_seconds_if_validated'])
    events = []
    for item in json.loads((ROOT / json_path[rid]).read_text(encoding='utf-8')):
        notes, starts, ends = item.get('note', []), item.get('note_start', []), item.get('note_end', [])
        if not len(notes) == len(starts) == len(ends):
            continue
        for p, s, e in zip(notes, starts, ends):
            if p == 0 or not (0 < p <= 127 and int(p) == p and 0 <= s < e <= duration + 0.001):
                continue
            events.append((float(s), float(e), int(p)))
    return sorted(events)

def reference_table(recording_ids):
    return pd.DataFrame([dict(note_id=f'{rid}:ref{i:05d}', recording_id=rid, start=s, end=e, pitch=p)
                         for rid in recording_ids for i, (s, e, p) in enumerate(parse_reference(rid))])

# Check 1: our parser reproduces the shared TRAINING answer key exactly (on 20 training recordings)
check_ids = sorted(full_references.recording_id.unique())[:20]
mine = reference_table(check_ids)[['recording_id','start','end','pitch']].reset_index(drop=True)
theirs = full_references[full_references.recording_id.isin(check_ids)].sort_values(['recording_id','start','end','pitch'])[['recording_id','start','end','pitch']].reset_index(drop=True)
pd.testing.assert_frame_equal(mine.sort_values(['recording_id','start','end','pitch']).reset_index(drop=True), theirs, check_dtype=False, atol=1e-6)
print('Check 1 passed: parser matches references_train.csv')

val_references = reference_table(val_manifest.recording_id)
val_unchanged = val_notes[cu.NOTE_COLUMNS].copy()

# Check 2: unchanged Basic Pitch on validation should reproduce the README's 0.2726
no_change = cu.evaluate_end_to_end(val_unchanged, val_references, val_manifest)['summary']
print(f"Check 2: validation no-change F1 = {no_change['song_macro_f1']:.4f} (README: 0.2726)")

val_versions = {name: apply_merges(val_unchanged, val_pairs[pred == 1]) for name, pred in val_pred.items()}
train_refs_backup = ref_by_rec       # oracle labels use the SAME labelling rule with validation answers (upper bound only)
ref_by_rec = {rid: g[['start','end','pitch']].to_numpy(float) for rid, g in val_references.groupby('recording_id')}
val_lookup = val_notes.set_index('note_id')
val_pairs['oracle_label'] = [merge_label(val_lookup.loc[a], val_lookup.loc[b]) for a, b in zip(val_pairs.left_note_id, val_pairs.right_note_id)]
ref_by_rec = train_refs_backup
val_versions['Oracle (perfect merging, upper bound)'] = apply_merges(val_unchanged, val_pairs[val_pairs.oracle_label == 1])

rows_out = [dict(version='No change', notes=len(val_unchanged), F1=no_change['song_macro_f1'],
                 F1_with_offsets=no_change['song_macro_offset_f1'], precision=no_change['song_macro_precision'], recall=no_change['song_macro_recall'])]
for name, table in val_versions.items():
    s = cu.evaluate_end_to_end(table, val_references, val_manifest)['summary']
    rows_out.append(dict(version=name, notes=len(table), F1=s['song_macro_f1'], F1_with_offsets=s['song_macro_offset_f1'],
                         precision=s['song_macro_precision'], recall=s['song_macro_recall']))
    print('scored:', name)
val_results = pd.DataFrame(rows_out).round(4)
val_results['change_vs_no_change'] = (val_results.F1 - val_results.F1.iloc[0]).round(4)
display(val_results)

stamp = time.strftime('%Y%m%d_%H%M')
val_results.to_csv(HERE / 'exports' / f'queenie_merge_v2_VALIDATION_results_{stamp}.csv', index=False)
for name, pred in val_pred.items():
    tag = name.split()[0].lower()
    cu.export_corrections(val_versions[name], val_manifest, HERE / 'exports' / f'queenie_merge_v2_VALIDATION_corrected_{tag}_{stamp}.csv')
    merge_actions(val_unchanged, val_pairs[pred == 1]).to_csv(HERE / 'exports' / f'queenie_merge_v2_VALIDATION_actions_{tag}_{stamp}.csv', index=False)
print('Saved validation results, corrected notes and action history for each frozen model.')

Check 1 passed: parser matches references_train.csv
Check 2: validation no-change F1 = 0.2726 (README: 0.2726)
scored: Rule baseline
scored: Logistic regression (threshold 0.6)
scored: Random forest (threshold 0.5)
scored: Oracle (perfect merging, upper bound)


,version,notes,F1,F1_with_offsets,precision,recall,change_vs_no_change
0,No change,2329,0.2726,0.1602,0.2830,0.2694,0.0000
1,Rule baseline,2105,0.2625,0.1491,0.2863,0.2479,-0.0101
2,Logistic regression (threshold 0.6),2294,0.2707,0.1591,0.2829,0.2656,-0.0019
3,Random forest (threshold 0.5),2135,0.2624,0.1515,0.2839,0.2494,-0.0102
4,"Oracle (perfect merging, upper bound)",2281,0.2763,0.1679,0.2891,0.2704,0.0037


Saved validation results, corrected notes and action history for each frozen model.


In [42]:
# === TRAINING ONLY: could a narrower merge rule work? ===
# Part 1: where are real merges concentrated? (labels only, seconds)
lab = joined[joined.merge_label.notna()].copy()
lab['pitch_diff'] = lab.pitch_delta.abs().map({0: 'same pitch', 1: '1 semitone'})
lab['gap_range'] = pd.cut(lab.gap, [-0.02, 0.14, 0.16, 0.20, 0.25, 0.30])
display(lab.groupby(['pitch_diff', 'gap_range'], observed=True).merge_label
           .agg(pairs='count', merges='sum', merge_rate='mean').round(2))

# Part 2: score a few narrow FIXED rules on the full training transcription (about 1-2 minutes each)
narrow_rules = {
    '1 semitone, gap <= 0.14 s': (joined.pitch_delta.abs() == 1) & (joined.gap <= 0.14),
    '1 semitone, gap <= 0.16 s': (joined.pitch_delta.abs() == 1) & (joined.gap <= 0.16),
    'same pitch, gap <= 0.16 s': (joined.pitch_delta.abs() == 0) & (joined.gap <= 0.16),
}
rows = []
for name, mask in narrow_rules.items():
    pred = mask.astype(int).to_numpy()
    m = cu.classification_metrics(y_all[is_labelled], pred[is_labelled])
    s = cu.evaluate_end_to_end(apply_merges(unchanged, joined[pred == 1]), full_references, full_manifest)['summary']
    rows.append(dict(rule=name, merges=int(pred.sum()), precision=round(m['precision'], 3),
                     F1=round(s['song_macro_f1'], 4), change_vs_no_change=round(s['song_macro_f1'] - 0.2628, 4)))
    print('scored:', name)
display(pd.DataFrame(rows))

pairs  merges  merge_rate
pitch_diff gap_range                               
1 semitone (-0.02, 0.14]    561   120.0        0.21
           (0.14, 0.16]      19     1.0        0.05
           (0.16, 0.2]       77     7.0        0.09
           (0.2, 0.25]       43     5.0        0.12
           (0.25, 0.3]       34     3.0        0.09
same pitch (-0.02, 0.14]     22     5.0        0.23
           (0.14, 0.16]      27     3.0        0.11
           (0.16, 0.2]       81    10.0        0.12
           (0.2, 0.25]       43     4.0        0.09
           (0.25, 0.3]       37     7.0        0.19

scored: 1 semitone, gap <= 0.14 s
scored: 1 semitone, gap <= 0.16 s
scored: same pitch, gap <= 0.16 s


,rule,merges,precision,F1,change_vs_no_change
0,"1 semitone, gap <= 0.14 s",730,0.214,0.2457,-0.0171
1,"1 semitone, gap <= 0.16 s",756,0.209,0.2451,-0.0177
2,"same pitch, gap <= 0.16 s",69,0.163,0.2625,-0.0003
